In [1]:
###########################################################
# Notebook 04 – Feature Engineering (Basis: 'price_totaal')
# Projekt: Preisbewertung NL
###########################################################

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor


In [3]:
#################################################
# 4.1. Vorbereitungen für Cross-Section Jahr 2023
#################################################

df_master = pd.read_csv("../data_clean/df_master.csv")
df_geo = pd.read_csv("../data_clean/df_geo.csv")

# Cross-Section für das Jahr 2023
df_model = df_master[df_master['year'] == 2023].copy()

# df_geo ebenfalls auf 2023 einschränken und pro Gemeinde nur eine Zeile
df_geo_2023 = df_geo[df_geo["year"] == 2023].drop_duplicates(subset="municipality_code")

# Koordinaten aus df_geo_2023 ergänzen (falls noch nicht enthalten)
if "longitude" not in df_model.columns:
    df_model = df_model.merge(
        df_geo_2023[["municipality_code", "longitude", "latitude"]],
        on="municipality_code",
        how="left"
    )

print(df_model.shape)
print()
df_model.head()


(341, 50)



,municipality_code,year,price_total,price_egw,price_mgw,ratio_price_model,ratio_price_realized,share_egw,share_mgw,housing_stock_egw_x,...,population_dec31,migration_net,migration_per_1000,migration_rate,avg_space_egw,avg_space_mgw,province,ratio_egw_mgw,longitude,latitude
0,GM0014,2023,349931.0,417685.371080,306594.600316,1.362338,1.362338,0.390099,0.609901,47871.0,...,243768.0,5621.0,23.603069,2.360307,0.990298,0.985892,Groningen,1.362338,6.620667,53.218625
1,GM0034,2023,424531.0,495255.358010,251950.903571,1.965682,1.965682,0.709317,0.290683,64328.0,...,226500.0,3675.0,16.492763,1.649276,0.990195,0.962468,Flevoland,1.965682,5.246608,52.367195
2,GM0037,2023,265130.0,302787.188580,127365.718020,2.377305,2.377305,0.785333,0.214667,11919.0,...,32013.0,-122.0,-3.796484,-0.379648,0.999916,1.002462,Groningen,2.377305,7.010322,53.002171
3,GM0047,2023,269888.0,309491.714084,124242.565495,2.491028,2.491028,0.786214,0.213786,10231.0,...,27520.0,-96.0,-3.476246,-0.347625,0.995233,0.991800,Groningen,2.491028,6.880092,53.088990
4,GM0050,2023,390515.0,437110.222031,149609.096121,2.921682,2.921682,0.837930,0.162070,7838.0,...,23899.0,207.0,8.737126,0.873713,0.984055,0.866286,Flevoland,2.921682,5.520536,52.319162


In [4]:
################################################
# 4.2. Zielvariable definieren (log_price_total)
################################################

import numpy as np

df_model['log_price_total'] = np.log(df_model['price_total'])
df_model['log_price_egw'] = np.log(df_model['price_egw'])
df_model['log_price_mgw'] = np.log(df_model['price_mgw'])


In [5]:
#################################################
# 4.3. Provinzen für One-Hot Encoding vorbereiten
#################################################

df_model['province_original'] = df_model['province']


In [6]:
#####################################
# 4.4. One-Hot Encoding der Provinzen
#####################################

df_model = pd.get_dummies(df_model, columns=['province'], drop_first=True)


In [7]:
df_model[['gemeentenaam'] + [col for col in df_model.columns if col.startswith('province_')]].head(10)


,gemeentenaam,province_original,province_Flevoland,province_Fryslân,province_Gelderland,province_Groningen,province_Limburg,province_Noord-Brabant,province_Noord-Holland,province_Overijssel,province_Utrecht,province_Zeeland,province_Zuid-Holland
0,Groningen (gemeente),Groningen,False,False,False,True,False,False,False,False,False,False,False
1,Almere,Flevoland,True,False,False,False,False,False,False,False,False,False,False
2,Stadskanaal,Groningen,False,False,False,True,False,False,False,False,False,False,False
3,Veendam,Groningen,False,False,False,True,False,False,False,False,False,False,False
4,Zeewolde,Flevoland,True,False,False,False,False,False,False,False,False,False,False
5,Achtkarspelen,Fryslân,False,True,False,False,False,False,False,False,False,False,False
6,Ameland,Fryslân,False,True,False,False,False,False,False,False,False,False,False
7,Harlingen,Fryslân,False,True,False,False,False,False,False,False,False,False,False
8,Heerenveen,Fryslân,False,True,False,False,False,False,False,False,False,False,False
9,Leeuwarden,Fryslân,False,True,False,False,False,False,False,False,False,False,False


In [8]:
df_model[['gemeentenaam'] + [col for col in df_model.columns if col.startswith('province_')]].head(10)


,gemeentenaam,province_original,province_Flevoland,province_Fryslân,province_Gelderland,province_Groningen,province_Limburg,province_Noord-Brabant,province_Noord-Holland,province_Overijssel,province_Utrecht,province_Zeeland,province_Zuid-Holland
0,Groningen (gemeente),Groningen,False,False,False,True,False,False,False,False,False,False,False
1,Almere,Flevoland,True,False,False,False,False,False,False,False,False,False,False
2,Stadskanaal,Groningen,False,False,False,True,False,False,False,False,False,False,False
3,Veendam,Groningen,False,False,False,True,False,False,False,False,False,False,False
4,Zeewolde,Flevoland,True,False,False,False,False,False,False,False,False,False,False
5,Achtkarspelen,Fryslân,False,True,False,False,False,False,False,False,False,False,False
6,Ameland,Fryslân,False,True,False,False,False,False,False,False,False,False,False
7,Harlingen,Fryslân,False,True,False,False,False,False,False,False,False,False,False
8,Heerenveen,Fryslân,False,True,False,False,False,False,False,False,False,False,False
9,Leeuwarden,Fryslân,False,True,False,False,False,False,False,False,False,False,False


In [9]:
###########################################
# 4.5. Lagevariable „log_density“ erstellen
###########################################

df_model['log_density'] = np.log(df_model['bevolkingsdichtheid'] + 1)


In [10]:
#################################
# 4.6. 'Randstad'-Dummy erstellen
#################################

randstad_provinces = ['Noord-Holland', 'Zuid-Holland', 'Utrecht', 'Flevoland']

df_model['is_randstad'] = df_model['province_original'].isin(randstad_provinces).astype(int)


In [11]:
province_dummies = [col for col in df_model.columns if col.startswith('province_')]

df_model = df_model.drop(columns=['province_original'], errors='ignore')


In [12]:
#############################
# 4.7. Skalierung durchführen
#############################

from sklearn.preprocessing import StandardScaler

# Liste der numerischen Variablen, die wir skalieren wollen
num_features = [
    'income_mean',
    'woz_mean',
    'ratio_price_model',
    'ratio_stock',
    'woningvoorraad_total',
    'pipeline_total',
    'new_build_stock_tot',
    'households',
    'log_density',
    'migration_per_1000',
    'employment_rate',
    'labor_participation_net'
]

# Skalierer initialisieren
scaler = StandardScaler()

# Skalierung anwenden
df_model[num_features] = scaler.fit_transform(df_model[num_features])


In [13]:
print(df_model[num_features].head())
print()
df_model[num_features].describe()


   income_mean  woz_mean  ratio_price_model  ratio_stock  \
0    -1.466662 -0.862329          -1.411029    -1.517418   
1    -0.281377 -0.108316          -0.792854    -0.779786   
2    -0.983768 -1.430571          -0.371113    -0.280731   
3    -0.910602 -1.758403          -0.254595    -0.272869   
4    -0.061880  0.219516           0.186645     0.338604   

   woningvoorraad_total  pipeline_total  new_build_stock_tot  households  \
0              2.611084        1.303138             2.944820    2.949466   
1              1.779708        1.699302             3.039765    1.754643   
2             -0.231567       -0.280865            -0.420107   -0.238352   
3             -0.285897       -0.198358            -0.408155   -0.285317   
4             -0.373843       -0.229134            -0.109707   -0.345423   

   log_density  migration_per_1000  employment_rate  labor_participation_net  
0     0.847414            2.500631        -3.652659                -0.815182  
1     1.132497          

,income_mean,woz_mean,ratio_price_model,ratio_stock,woningvoorraad_total,pipeline_total,new_build_stock_tot,households,log_density,migration_per_1000,employment_rate,labor_participation_net
count,3.410000e+02,3.410000e+02,3.410000e+02,3.410000e+02,3.410000e+02,3.410000e+02,3.410000e+02,3.410000e+02,3.410000e+02,3.410000e+02,3.410000e+02,3.410000e+02
mean,-1.588824e-16,2.604629e-16,-1.041852e-16,2.083703e-16,5.209258e-17,3.125555e-17,-1.041852e-17,1.041852e-17,-2.083703e-17,4.427869e-17,-1.210631e-14,-4.584147e-16
std,1.001470e+00,1.001470e+00,1.001470e+00,1.001470e+00,1.001470e+00,1.001470e+00,1.001470e+00,1.001470e+00,1.001470e+00,1.001470e+00,1.001470e+00,1.001470e+00
min,-1.466662e+00,-1.944174e+00,-1.743192e+00,-1.720184e+00,-6.108388e-01,-3.391434e-01,-4.861702e-01,-5.837797e-01,-2.998616e+00,-4.172175e+00,-4.737072e+00,-5.140665e+00
25%,-4.935577e-01,-6.547024e-01,-8.040727e-01,-7.981666e-01,-3.696734e-01,-2.677683e-01,-3.248290e-01,-3.589290e-01,-7.343519e-01,-5.920328e-01,-3.994201e-01,-5.563922e-01
50%,-1.204124e-01,-1.083159e-01,-1.375615e-01,-1.398888e-01,-2.406870e-01,-2.068703e-01,-2.335351e-01,-2.463820e-01,-1.040574e-01,-5.400442e-02,2.512276e-01,1.460367e-01
75%,2.600495e-01,4.599260e-01,6.349743e-01,6.668790e-01,-8.641884e-03,-5.757206e-02,2.109178e-02,-2.104469e-02,7.691338e-01,3.787557e-01,6.849928e-01,6.636158e-01
max,1.122759e+01,6.033068e+00,3.579097e+00,3.627946e+00,1.190305e+01,1.395682e+01,1.430311e+01,1.193863e+01,2.461277e+00,5.003136e+00,1.552523e+00,2.918782e+00


In [14]:
##############################
# 4.8. Feature‑Satz definieren
##############################

# Zielvariable festlegen

target = 'log_price_total'

# Numerische Features
num_features = [
    'income_mean',
    'woz_mean',
    'log_density',
    'migration_per_1000',
    'employment_rate',
    'labor_participation_net'
]

# Lagevariablen
location_features = [
    'longitude',
    'latitude',
    'is_randstad'
]

# Sonstige Variablen
additional_features = [
    'ratio_price_model',
    'ratio_stock',
    'woningvoorraad_total',
    'pipeline_total',
    'new_build_stock_tot',
    'households'
]

# Provinz-Dummies
province_dummies = [col for col in df_model.columns if col.startswith('province_')]

# Finaler Feature-Satz
feature_cols = num_features + location_features + province_dummies + additional_features
len(feature_cols), feature_cols


(26,
 ['income_mean',
  'woz_mean',
  'log_density',
  'migration_per_1000',
  'employment_rate',
  'labor_participation_net',
  'longitude',
  'latitude',
  'is_randstad',
  'province_Flevoland',
  'province_Fryslân',
  'province_Gelderland',
  'province_Groningen',
  'province_Limburg',
  'province_Noord-Brabant',
  'province_Noord-Holland',
  'province_Overijssel',
  'province_Utrecht',
  'province_Zeeland',
  'province_Zuid-Holland',
  'ratio_price_model',
  'ratio_stock',
  'woningvoorraad_total',
  'pipeline_total',
  'new_build_stock_tot',
  'households'])

In [15]:
# Dataframe Modell-Dataframe erstellen

X = df_model[feature_cols]
y = df_model[target]


In [16]:
df_model.head(20)

,municipality_code,year,price_total,price_egw,price_mgw,ratio_price_model,ratio_price_realized,share_egw,share_mgw,housing_stock_egw_x,...,province_Groningen,province_Limburg,province_Noord-Brabant,province_Noord-Holland,province_Overijssel,province_Utrecht,province_Zeeland,province_Zuid-Holland,log_density,is_randstad
0,GM0014,2023,349931.0,417685.371080,306594.600316,-1.411029,1.362338,0.390099,0.609901,47871.0,...,True,False,False,False,False,False,False,False,0.847414,0
1,GM0034,2023,424531.0,495255.358010,251950.903571,-0.792854,1.965682,0.709317,0.290683,64328.0,...,False,False,False,False,False,False,False,False,1.132497,1
2,GM0037,2023,265130.0,302787.188580,127365.718020,-0.371113,2.377305,0.785333,0.214667,11919.0,...,True,False,False,False,False,False,False,False,-0.645781,0
3,GM0047,2023,269888.0,309491.714084,124242.565495,-0.254595,2.491028,0.786214,0.213786,10231.0,...,True,False,False,False,False,False,False,False,-0.368696,0
4,GM0050,2023,390515.0,437110.222031,149609.096121,0.186645,2.921682,0.837930,0.162070,7838.0,...,False,False,False,False,False,False,False,False,-1.649128,1
5,GM0059,2023,319773.0,352514.503328,90390.506978,1.188914,3.899906,0.875092,0.124908,10754.0,...,False,False,False,False,False,False,False,False,-0.638753,0
6,GM0060,2023,353454.0,402270.972301,167165.398446,-0.341278,2.406425,0.792361,0.207639,1473.0,...,False,False,False,False,False,False,False,False,-1.978229,0
7,GM0072,2023,298340.0,342642.418554,152994.079140,-0.512224,2.239580,0.766397,0.233603,6158.0,...,False,False,False,False,False,False,False,False,0.188889,0
8,GM0074,2023,362420.0,422027.973341,172779.028738,-0.304226,2.442588,0.760850,0.239150,18268.0,...,False,False,False,False,False,False,False,False,-0.649313,0
9,GM0080,2023,294731.0,356275.851085,188394.640460,-0.869254,1.891115,0.633402,0.366598,40971.0,...,False,False,False,False,False,False,False,False,0.002564,0


In [17]:
df_model[['gemeentenaam'] + [col for col in df_model.columns if col.startswith('province_')]].head(10)


,gemeentenaam,province_Flevoland,province_Fryslân,province_Gelderland,province_Groningen,province_Limburg,province_Noord-Brabant,province_Noord-Holland,province_Overijssel,province_Utrecht,province_Zeeland,province_Zuid-Holland
0,Groningen (gemeente),False,False,False,True,False,False,False,False,False,False,False
1,Almere,True,False,False,False,False,False,False,False,False,False,False
2,Stadskanaal,False,False,False,True,False,False,False,False,False,False,False
3,Veendam,False,False,False,True,False,False,False,False,False,False,False
4,Zeewolde,True,False,False,False,False,False,False,False,False,False,False
5,Achtkarspelen,False,True,False,False,False,False,False,False,False,False,False
6,Ameland,False,True,False,False,False,False,False,False,False,False,False
7,Harlingen,False,True,False,False,False,False,False,False,False,False,False
8,Heerenveen,False,True,False,False,False,False,False,False,False,False,False
9,Leeuwarden,False,True,False,False,False,False,False,False,False,False,False


In [18]:
# Dataframes als csv-Datei speichern

df_model.to_csv('../data_clean/df_model_prepared.csv', index=False)
X.to_csv('../data_clean/X_features.csv', index=False)
y.to_csv('../data_clean/y_target.csv', index=False)


In [19]:
########
# Exkurs
########

df_model.columns

Index(['municipality_code', 'year', 'price_total', 'price_egw', 'price_mgw',
       'ratio_price_model', 'ratio_price_realized', 'share_egw', 'share_mgw',
       'housing_stock_egw_x', 'housing_stock_mgw_x', 'ratio_area',
       'ratio_stock', 'gemeentenaam', 'population', 'bevolkingsdichtheid',
       'households', 'woningvoorraad_total', 'housing_stock_egw_y',
       'housing_stock_mgw_y', 'living_space_total', 'living_space_egw',
       'living_space_mgw', 'income_mean', 'income_median', 'woz_mean',
       'pipeline_total', 'pipeline_started', 'pipeline_permit',
       'new_build_stock_tot', 'new_build_liv_space_tot', 'new_build_stock_egw',
       'new_build_liv_space_egw', 'new_build_stock_mgw',
       'new_build_liv_space_mgw', 'unemployment_rate',
       'labor_participation_gross', 'labor_participation_net',
       'employment_rate', 'population_jan1', 'population_dec31',
       'migration_net', 'migration_per_1000', 'migration_rate',
       'avg_space_egw', 'avg_space_mgw', 'ra

In [20]:
province_cols = [col for col in df_model.columns if col.startswith('province_')]

df_model[province_cols].sum().sort_values(ascending=False)


province_Noord-Brabant    56
province_Zuid-Holland     50
province_Gelderland       50
province_Noord-Holland    44
province_Limburg          31
province_Utrecht          26
province_Overijssel       25
province_Fryslân          18
province_Zeeland          13
province_Groningen        10
province_Flevoland         6
dtype: int64

In [21]:
df_model[province_cols].sum(axis=1).value_counts()


1    329
0     12
Name: count, dtype: int64